# Smoke test S_0 → S_1 trên D_dev (Task.md mục 0.1)

Notebook này chạy toàn bộ smoke test trên Google Colab. Hướng dẫn chi tiết: `Instruction_0.1.md` (mục "Chạy trên Colab").

**Trước khi chạy:**
1. `Runtime → Change runtime type → GPU` (T4 miễn phí là đủ; L4/A100 nhanh hơn và dùng bf16 đúng như config).
2. Tải `smoke_bundle.zip` lên Google Drive vào thư mục `MyDrive/DACNTT_smoke/`.
3. Chạy lần lượt từng ô. Mọi kết quả và adapter được ghi thẳng vào Drive, nên nếu Colab bị ngắt thì chạy lại từ đầu: các bước đã xong sẽ được bỏ qua.

**Không dùng D_test ở bất kỳ bước nào.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## 1. Kết nối Drive và giải nén bundle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_DIR = '/content/drive/MyDrive/DACNTT_smoke'      # đổi nếu bạn để bundle ở chỗ khác
BUNDLE = f'{DRIVE_DIR}/smoke_bundle.zip'
RESULTS = f'{DRIVE_DIR}/results'                         # = reports/task5/smoke_test trong repo
ADAPTERS = f'{DRIVE_DIR}/adapters'                       # trọng số LoRA (không commit vào git)
WORK = '/content/manual'                                 # = experiments/week2September/manual
os.makedirs(RESULTS, exist_ok=True); os.makedirs(ADAPTERS, exist_ok=True)
assert os.path.exists(BUNDLE), f'Không thấy {BUNDLE}'

In [ ]:
!rm -rf {WORK} && mkdir -p {WORK} && unzip -q -o "{BUNDLE}" -d {WORK}
%cd {WORK}
!sha256sum -c MANIFEST.sha256
# kết quả ghi thẳng vào Drive
!mkdir -p reports/task5 && rm -rf reports/task5/smoke_test && ln -s "{RESULTS}" reports/task5/smoke_test
!ls -la reports/task5/

## 2. Cài thư viện

Ghim đúng phiên bản trong `requirements-lock.txt` của repo cho các thư viện ảnh hưởng tới kết quả. Giữ nguyên torch có sẵn của Colab; phiên bản thực tế được ghi vào `environment_colab.txt`.

In [ ]:
!pip install -q transformers==5.15.0 peft==0.20.0 accelerate==1.14.0 scipy==1.17.1
# Colab cài sẵn torchao cũ (0.10), peft 0.20 từ chối nó khi tạo LoRA; dự án không dùng torchao
!pip uninstall -y -q torchao
!python -c "import torch, transformers, peft, scipy; print(torch.__version__, transformers.__version__, peft.__version__, scipy.__version__)"
!pip freeze > "{RESULTS}/environment_colab.txt" && nvidia-smi >> "{RESULTS}/environment_colab.txt"
# Nếu tải model bị treo, bỏ dấu # ở dòng dưới rồi chạy lại các bước sau:
# os.environ['HF_HUB_DISABLE_XET'] = '1'

## 3. Tham số theo GPU

- GPU hỗ trợ bf16 (L4, A100): chạy bf16 như config.
- T4: không có bf16, nên script tự dùng fp32 (tránh fp16 vì Qwen2.5 có thể tràn số). Chậm hơn nhưng an toàn.

Nếu gặp lỗi hết bộ nhớ (CUDA out of memory), giảm `EVAL_BS` hoặc `MICRO_BS` rồi chạy lại ô bị lỗi.

In [ ]:
BF16 = torch.cuda.is_bf16_supported()
EVAL_BS = 32 if BF16 else 16      # batch khi sinh lời giải
MICRO_BS = 4 if BF16 else 2       # micro-batch khi huấn luyện (effective batch = 8 theo config)
NLL_BS = 8 if BF16 else 4
SEEDS = [13, 42]
print(dict(BF16=BF16, EVAL_BS=EVAL_BS, MICRO_BS=MICRO_BS, NLL_BS=NLL_BS, SEEDS=SEEDS))

## 4. Bước A: đánh giá S_0 trên D_dev (500 câu)

Nếu bị ngắt giữa chừng, chạy lại ô này: các câu đã có kết quả sẽ được bỏ qua.

In [ ]:
!python scripts/eval_student.py --tag S0 --batch-size {EVAL_BS} --out reports/task5/smoke_test/eval_S0_dev.jsonl

Kiểm tra nhanh trước khi đi tiếp: accuracy S_0 và **tỉ lệ đúng định dạng** (`format_compliance`). Nếu tỉ lệ đúng định dạng thấp, phần lớn thay đổi sau huấn luyện có thể chỉ là học định dạng: so với `lenient_accuracy_exploratory`.

In [ ]:
import json
s = json.load(open('reports/task5/smoke_test/eval_S0_dev.summary.json'))
{k: s[k] for k in ['accuracy', 'format_compliance', 'lenient_accuracy_exploratory', 'mean_output_tokens', 'n_hit_max_tokens', 'elapsed_seconds_this_run', 'peak_gpu_memory_gb', 'dtype']}

## 5. Bước B: chọn 200 trajectory cho mỗi seed

Chỉ trong smoke test: bỏ `n_cap = 1` vì pool sơ bộ chỉ có 98 câu.

In [ ]:
for seed in SEEDS:
    !python scripts/select_smoke_train_set.py --seed {seed}

## 6. Bước B–C: huấn luyện 1 vòng rồi đánh giá S_1, cho từng seed

In [ ]:
import os, shutil
for seed in SEEDS:
    adapter = f'{ADAPTERS}/student_ckpt_001_smoke_seed{seed}'
    if os.path.exists(f'{adapter}/train_log.json'):
        print(f'seed {seed}: adapter đã có, bỏ qua huấn luyện')
    else:
        !python scripts/train_round.py --seed {seed} --micro-batch {MICRO_BS} --train-set reports/task5/smoke_test/train_set_seed{seed}.json --adapter-dir "{adapter}"
    assert os.path.exists(f'{adapter}/train_log.json'), f'seed {seed}: huấn luyện lỗi, xem log ở trên'
    shutil.copy(f'{adapter}/train_log.json', f'reports/task5/smoke_test/train_log_seed{seed}.json')
    !python scripts/eval_student.py --tag S1_seed{seed} --adapter "{adapter}" --batch-size {EVAL_BS} --out reports/task5/smoke_test/eval_S1_dev_seed{seed}.jsonl

## 7. Bước D: NLL của pool (480 trajectory) tại S_0 và S_1

In [ ]:
import os
if not os.path.exists('reports/task5/smoke_test/nll_pool_S0.summary.json'):
    !python scripts/compute_nll.py --tag S0 --batch-size {NLL_BS} --out reports/task5/smoke_test/nll_pool_S0.jsonl --pool trajectory/pool/batch_100_teacher_verified.jsonl
for seed in SEEDS:
    if not os.path.exists(f'reports/task5/smoke_test/nll_pool_S1_seed{seed}.summary.json'):
        adapter = f'{ADAPTERS}/student_ckpt_001_smoke_seed{seed}'
        !python scripts/compute_nll.py --tag S1_seed{seed} --adapter "{adapter}" --batch-size {NLL_BS} --out reports/task5/smoke_test/nll_pool_S1_seed{seed}.jsonl --pool trajectory/pool/batch_100_teacher_verified.jsonl

## 8. Phân tích và quyết định theo ngưỡng đã chốt

In [ ]:
!python scripts/analyze_smoke_test.py --seeds {' '.join(map(str, SEEDS))}

## 9. Mang kết quả về repo

Tải thư mục `MyDrive/DACNTT_smoke/results/` về và chép vào `experiments/week2September/manual/reports/task5/smoke_test/` trong repo. **Không** chép thư mục `adapters/` vào repo (trọng số model không được commit).

Sau đó viết `smoke_test_report.md` và cập nhật `task5_decisions.md`, config (v2 nếu đổi tham số), SHA256, changelog theo `Instruction_0.1.md` mục 9.

In [ ]:
!ls -la "{RESULTS}"